# Orchestrating the Complete Claim Pipeline

This notebook is the **orchestration fundamentals** notebook. It assumes you
have already worked through `03_tool_calling_notebook.ipynb`, which defined:

- All the `[TOOL]` functions (deterministic)
- `run_react_agent` (the hand-rolled ReAct loop) and the three `[AGENT]`
  wrappers: `extract_diagnosis_and_treatment`,
  `check_diagnosis_treatment_consistency`, `check_eligibility`

Here we write the **controller** — plain Python, no LLM — that decides, at
each stage of the pipeline, whether to call a tool directly or hand off to
an agent, and that implements the *branching logic* real systems need:
what happens if the doctor hasn't signed off yet? What if the agent isn't
confident? What if the bill exceeds what's eligible?


In [ ]:
%run tool_calling_notebook.ipynb

## 1. The orchestration pattern

Every step below is annotated with whether it's a **direct function call**
or an **agent invocation**, matching the classification in
`01_case_study_and_functions.md`. The orchestrator's job is to:

1. Call the right thing (tool vs. agent) at the right time.
2. Pass the output of one step as the input to the next.
3. Branch: stop and escalate (`flag_for_manual_review`) instead of
   ploughing ahead when something is missing or the agent isn't confident.
4. Assemble a final, structured result.

This is intentionally a plain sequential pipeline with a few `if` branches —
not a generic "agent decides everything" loop — because most of this
workflow genuinely has a fixed, known order. Reserving free-form agent
autonomy for only the steps that need it is itself a design lesson.


In [ ]:
def orchestrate_claim_process(patient_id: str, verbose: bool = True) -> dict:
    """[CONTROLLER] Run the full discharge-to-claim pipeline for one patient.

    Returns a dict describing the outcome: either a completed claim + bill,
    or a manual-review flag with a reason.
    """
    def log(msg):
        if verbose:
            print(msg)

    # --- Step 1: admission record --------------------------------- [TOOL]
    log("STEP 1 [TOOL] get_patient_record")
    patient_record = get_patient_record(patient_id)
    log(f"  -> {patient_record}")

    # --- Step 2: doctor sign-off check ------------------------------ [TOOL]
    log("STEP 2 [TOOL] get_doctor_signoff_status")
    signoff = get_doctor_signoff_status(patient_id)
    log(f"  -> {signoff}")
    if not signoff.get("signed_off"):
        reason = "Doctor has not signed off on this patient's record yet."
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    # --- Step 3: doctor's notes -------------------------------------- [TOOL]
    log("STEP 3 [TOOL] get_doctor_notes")
    notes = get_doctor_notes(patient_id)
    log(f"  -> {notes[:80]}...")

    # --- Step 4: extract diagnosis & treatment ----------------------- [AGENT]
    log("STEP 4 [AGENT] extract_diagnosis_and_treatment")
    extraction = extract_diagnosis_and_treatment(notes)
    log(f"  -> {extraction}")
    if not extraction.get("icd_code") or not extraction.get("cpt_code"):
        reason = f"Could not confidently extract/validate codes: {extraction}"
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    diagnosis_code = extraction["icd_code"]
    procedure_code = extraction["cpt_code"]

    # --- Step 5: clinical consistency check -------------------------- [AGENT]
    log("STEP 5 [AGENT] check_diagnosis_treatment_consistency")
    consistency = check_diagnosis_treatment_consistency(
        extraction["diagnosis"], extraction["treatment"]
    )
    log(f"  -> {consistency}")
    if consistency.get("needs_human_review") or not consistency.get("consistent"):
        reason = f"Diagnosis/treatment consistency check flagged this: {consistency}"
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    # --- Step 6: insurance policy ------------------------------------- [TOOL]
    log("STEP 6 [TOOL] get_insurance_policy")
    policy = get_insurance_policy(patient_record["insurance_id"])
    log(f"  -> {policy}")

    # --- Step 7: eligibility check -------------------------------------- [AGENT]
    log("STEP 7 [AGENT] check_eligibility")
    eligibility = check_eligibility(
        policy, diagnosis_code, procedure_code, patient_record["days_admitted"]
    )
    log(f"  -> {eligibility}")
    if not eligibility.get("eligible"):
        reason = f"Claim is not eligible under this policy: {eligibility.get('reasoning')}"
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    # --- Step 8: hospital bill ------------------------------------------ [TOOL]
    log("STEP 8 [TOOL] get_hospital_bill_items + calculate_total_bill")
    bill_items = get_hospital_bill_items(patient_id)
    total_bill = calculate_total_bill(bill_items)
    log(f"  -> total_bill = {total_bill}")

    # --- Step 9: money math ---------------------------------------------- [TOOL]
    log("STEP 9 [TOOL] calculate_covered_amount + calculate_patient_payable")
    covered_amount = calculate_covered_amount(
        policy, total_bill, patient_record["days_admitted"], eligibility
    )
    patient_payable = calculate_patient_payable(total_bill, covered_amount)
    log(f"  -> covered_amount = {covered_amount}, patient_payable = {patient_payable}")

    # --- Step 10: generate outputs ---------------------------------------- [TOOL]
    log("STEP 10 [TOOL] generate_claim_form + generate_discharge_bill")
    claim_form = generate_claim_form(
        patient_record, diagnosis_code, procedure_code, covered_amount,
        eligibility.get("reasoning", ""),
    )
    discharge_bill = generate_discharge_bill(
        patient_record, total_bill, covered_amount, patient_payable
    )

    return {
        "status": "completed",
        "patient_record": patient_record,
        "diagnosis": extraction["diagnosis"],
        "treatment": extraction["treatment"],
        "diagnosis_code": diagnosis_code,
        "procedure_code": procedure_code,
        "eligibility": eligibility,
        "total_bill": total_bill,
        "covered_amount": covered_amount,
        "patient_payable": patient_payable,
        "claim_form": claim_form,
        "discharge_bill": discharge_bill,
    }


## Compare the flow diagram with the above mentioned flow and figure out if there is any differences?

## 2. Run it end to end

This mirrors the worked example in `02_react_transcript.md`, but now the
[AGENT] steps are real LLM calls (via `run_react_agent` from Notebook 1)
rather than a hand-written transcript.


In [ ]:
result = orchestrate_claim_process("P1042")
result


## 3. A patient whose claim should be escalated

`P2091`'s bill (a coronary angioplasty) is far larger. Try it and see how
the eligibility cap and covered-amount math behave once the total bill
exceeds the eligible amount — this branch needs *no* extra LLM reasoning,
because `calculate_covered_amount` already knows how to cap deterministically
once eligibility has been decided.


In [ ]:
result_2091 = orchestrate_claim_process("P2091")
result_2091


## 4. Forcing an escalation path

To see the manual-review branch fire, temporarily edit `DOCTOR_SIGNOFF` (in
Notebook 1's mock data) so a patient's sign-off is `False`, then re-run
`orchestrate_claim_process` for that patient. Confirm the pipeline stops at
Step 2 and never wastes an LLM call on Steps 4, 5, or 7 — this is the
benefit of putting the cheap, deterministic checks *before* the expensive,
LLM-backed ones in the pipeline order.


In [ ]:
# Example (uncomment to try):
# DOCTOR_SIGNOFF["P1042"] = {"signed_off": False, "signed_by": None, "timestamp": None}
# orchestrate_claim_process("P1042")


## 5. Design takeaways for students

1. **Order matters for cost and speed.** Cheap deterministic checks
   (sign-off, record lookups) run before expensive LLM-backed reasoning
   steps, so failures are caught early and cheaply.
2. **The orchestrator is not itself an agent.** It's a fixed, auditable
   sequence of steps with explicit branches — easy to test, log, and debug.
   Only the three genuinely ambiguous sub-steps are delegated to an LLM
   ReAct loop.
3. **Escalation is a first-class outcome**, not an error. A well-designed
   pipeline for a regulated domain like insurance claims should default to
   *asking a human* whenever an agent's confidence, or a policy's clarity,
   is in question — not to silently guess.
4. **Determinism after judgment.** Once eligibility (a judgment call) is
   decided, everything downstream (money math, document generation) is
   deterministic and reproducible — which matters a lot when the output is
   a legal/financial document like an insurance claim.

## 6. Exercises

1. Add a Step 0 that checks `days_admitted <= 0` or a missing
   `insurance_id` and escalates immediately, before even checking sign-off.
2. Add a retry: if `extract_diagnosis_and_treatment` fails to resolve a code
   on the first try, re-run it once with an augmented prompt asking the
   model to consider common misspellings/synonyms, before escalating.
3. Instrument the orchestrator to also record, for each step, whether it
   was a `[TOOL]` or `[AGENT]` call and how long it took — then print a
   short summary table at the end of a run. This is the first building
   block of an observability/tracing layer for agentic pipelines.
4. Batch-run `orchestrate_claim_process` over all patient IDs in `PATIENTS`
   and produce a summary: how many completed straight through vs. how many
   were escalated, and why.
